In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:
#Menggunakan ecosystem dari Hugging Face Karena paling praktis

!pip install transformers datasets accelerate

In [4]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


***PROSES TRAINING***

***Dataset***

In [5]:
# Load Dataset

import json
import os

train_path = "/content/drive/MyDrive/Colab Notebooks/External Resource/Liputan6_data/canonical/train/"

def load_all_json(folder):
    data = []

    files = sorted(os.listdir(folder))[:1000]

    for filename in files:
        if filename.endswith(".json"):
            file_path = os.path.join(folder, filename)

            with open(file_path, 'r') as f:
                item = json.load(f)
                data.append(item)

    return data

train_data = load_all_json(train_path)

print("Jumlah data train_data:", len(train_data))
print("Contoh data train_data:", train_data[0])


Jumlah data train_data: 500
Contoh data train_data: {'id': 26408, 'url': 'https://www.liputan6.com/news/read/26408/pbb-siap-membantu-penyelesaian-konflik-ambon', 'clean_article': [['Liputan6', '.', 'com', ',', 'Ambon', ':', 'Partai', 'Bulan', 'Bintang', 'wilayah', 'Maluku', 'bertekad', 'membantu', 'pemerintah', 'menyelesaikan', 'konflik', 'di', 'provinsi', 'tersebut', '.'], ['Syaratnya', ',', 'penanganan', 'penyelesaian', 'konflik', 'Maluku', 'harus', 'dimulai', 'dari', 'awal', 'kerusuhan', ',', 'yakni', '19', 'Januari', '1999', '.'], ['Demikian', 'hasil', 'Musyawarah', 'Wilayah', 'I', 'PBB', 'Maluku', 'yang', 'dimulai', 'Sabtu', 'pekan', 'silam', 'dan', 'berakhir', 'Senin', '(', '31/12', ')', 'di', 'Ambon', '.'], ['Menurut', 'seorang', 'fungsionaris', 'PBB', 'Ridwan', 'Hasan', ',', 'persoalan', 'di', 'Maluku', 'bisa', 'selesai', 'asalkan', 'pemerintah', 'dan', 'aparat', 'keamanan', 'serius', 'menangani', 'setiap', 'persoalan', 'di', 'Maluku', 'secara', 'komprehensif', 'dan', 'bijaksan

In [6]:
# Convert Dataset Untuk mengambil clean_artikel dan clean_summary saja

def convert_to_text(data):
    new_data = []

    for item in data:
        # Gabungkan artikel
        article = " ".join(
            [" ".join(sentence) for sentence in item["clean_article"]]
        )

        # Gabungkan summary
        summary = " ".join(
            [" ".join(sentence) for sentence in item["clean_summary"]]
        )

        new_data.append({
            "clean_article": article,
            "clean_summary": summary
        })

    return new_data

train_data_convert = convert_to_text(train_data)

print(train_data[0])
print(train_data_convert[0])

{'id': 26408, 'url': 'https://www.liputan6.com/news/read/26408/pbb-siap-membantu-penyelesaian-konflik-ambon', 'clean_article': [['Liputan6', '.', 'com', ',', 'Ambon', ':', 'Partai', 'Bulan', 'Bintang', 'wilayah', 'Maluku', 'bertekad', 'membantu', 'pemerintah', 'menyelesaikan', 'konflik', 'di', 'provinsi', 'tersebut', '.'], ['Syaratnya', ',', 'penanganan', 'penyelesaian', 'konflik', 'Maluku', 'harus', 'dimulai', 'dari', 'awal', 'kerusuhan', ',', 'yakni', '19', 'Januari', '1999', '.'], ['Demikian', 'hasil', 'Musyawarah', 'Wilayah', 'I', 'PBB', 'Maluku', 'yang', 'dimulai', 'Sabtu', 'pekan', 'silam', 'dan', 'berakhir', 'Senin', '(', '31/12', ')', 'di', 'Ambon', '.'], ['Menurut', 'seorang', 'fungsionaris', 'PBB', 'Ridwan', 'Hasan', ',', 'persoalan', 'di', 'Maluku', 'bisa', 'selesai', 'asalkan', 'pemerintah', 'dan', 'aparat', 'keamanan', 'serius', 'menangani', 'setiap', 'persoalan', 'di', 'Maluku', 'secara', 'komprehensif', 'dan', 'bijaksana', '.'], ['Itulah', 'sebabnya', ',', 'PBB', 'wilaya

In [7]:
# Convert ke Format Dataset

from datasets import Dataset

train_dataset = Dataset.from_list(train_data_convert)

In [8]:
train_dataset

Dataset({
    features: ['clean_article', 'clean_summary'],
    num_rows: 500
})

In [9]:
# Load Model BERT
# Menggunakan facebook/bart-base

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [10]:
# Tokenisasi

def tokenize_function(example):
    inputs = tokenizer(
        example['clean_article'],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example['clean_summary'],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized_train = train_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [11]:
# Training Setup

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=100,
    save_steps=500,
    fp16=True
)


In [12]:
# Traininer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train
)

In [13]:
# Mulai Training

trainer.train()

Step,Training Loss
100,3.542982


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=125, training_loss=3.150939636230469, metrics={'train_runtime': 34.0361, 'train_samples_per_second': 14.69, 'train_steps_per_second': 3.673, 'total_flos': 152434114560000.0, 'train_loss': 3.150939636230469, 'epoch': 1.0})

In [14]:
# Simpan Model
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/Model/TextSummarization/Model_TextSummarization_v01_eko")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 ***Percobaan dengan data Dev***

In [15]:
dev_path = "/content/drive/MyDrive/Colab Notebooks/External Resource/Liputan6_data/canonical/dev/"

dev_data = load_all_json(dev_path)

print("Jumlah data dev_data:", len(dev_data))
print("Contoh data dev_data:", dev_data[0])

Jumlah data dev_data: 500
Contoh data dev_data: {'id': 1, 'url': 'https://www.liputan6.com/news/read/1/batas-penyerahan-aset-dua-pekan-lagi', 'clean_article': [['Liputan6', '.', 'com', ',', 'Jakarta', ':', 'Pemerintah', 'masih', 'memberikan', 'waktu', 'dua', 'minggu', 'lagi', 'kepada', 'seluruh', 'konglomerat', 'yang', 'telah', 'menandatangani', 'perjanjian', 'pengembalian', 'bantuan', 'likuiditas', 'Bank', 'Indonesia', 'dengan', 'jaminan', 'aset', '(', 'MSAA', ')', ',', 'untuk', 'secepatnya', 'menyerahkan', 'jaminan', 'pribadi', 'serta', 'aset', '.'], ['Jika', 'lewat', 'dari', 'tenggat', 'tersebut', ',', 'pemerintah', 'akan', 'menerapkan', 'tindakan', 'hukum', '.'], ['Hal', 'tersebut', 'dikemukakan', 'Menteri', 'Koordinator', 'Bidang', 'Perekonomian', 'Rizal', 'Ramli', 'di', 'Jakarta', ',', 'baru-baru', 'ini', '.'], ['Rizal', 'mengakui', 'bahwa', 'permintaan', 'untuk', 'meminta', 'jaminan', 'pribadi', 'atau', 'personal', 'guarantee', 'pada', 'awalnya', 'ditentang', 'sejumlah', 'konglo

In [16]:
dev_data_convert = convert_to_text(dev_data)

print(dev_data[0])
print(dev_data_convert[0])

{'id': 1, 'url': 'https://www.liputan6.com/news/read/1/batas-penyerahan-aset-dua-pekan-lagi', 'clean_article': [['Liputan6', '.', 'com', ',', 'Jakarta', ':', 'Pemerintah', 'masih', 'memberikan', 'waktu', 'dua', 'minggu', 'lagi', 'kepada', 'seluruh', 'konglomerat', 'yang', 'telah', 'menandatangani', 'perjanjian', 'pengembalian', 'bantuan', 'likuiditas', 'Bank', 'Indonesia', 'dengan', 'jaminan', 'aset', '(', 'MSAA', ')', ',', 'untuk', 'secepatnya', 'menyerahkan', 'jaminan', 'pribadi', 'serta', 'aset', '.'], ['Jika', 'lewat', 'dari', 'tenggat', 'tersebut', ',', 'pemerintah', 'akan', 'menerapkan', 'tindakan', 'hukum', '.'], ['Hal', 'tersebut', 'dikemukakan', 'Menteri', 'Koordinator', 'Bidang', 'Perekonomian', 'Rizal', 'Ramli', 'di', 'Jakarta', ',', 'baru-baru', 'ini', '.'], ['Rizal', 'mengakui', 'bahwa', 'permintaan', 'untuk', 'meminta', 'jaminan', 'pribadi', 'atau', 'personal', 'guarantee', 'pada', 'awalnya', 'ditentang', 'sejumlah', 'konglomerat', '.'], ['Sebab', 'para', 'debitor', 'meng

In [17]:
# dev_dataset = Dataset.from_list(dev_data_convert)
# msh blm tw kepakai atau ngga

In [19]:
# Evaluasi Model (PENTING BANGET)
# Menggunakan Rouge Score

!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=b69f90062edc0f7f09fff63497c2cf471c789fc43d34ad8222c44d2df2609750
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [21]:
# Evaluasi Model (PENTING BANGET)
# Menggunakan Rouge Score

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

pred = "hasil summary model"
ref = "summary asli"

score = scorer.score(ref, pred)
print(score)

{'rouge1': Score(precision=0.3333333333333333, recall=0.5, fmeasure=0.4), 'rougeL': Score(precision=0.3333333333333333, recall=0.5, fmeasure=0.4)}


In [23]:
# Test Summarization

text = dev_data_convert[0]['clean_article']
ref  = dev_data_convert[0]['clean_summary']

inputs = tokenizer(text, return_tensors="pt", truncation=True).to("cuda")

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=100,
    num_beams=4
)

pred = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

score = scorer.score(ref, pred)

print("ARTICLE:", text[:300])
print("\nPREDICT:", pred)
print("\nREFERENCE:", ref)
print("\nSCORE:", score)

ARTICLE: Liputan6 . com , Jakarta : Pemerintah masih memberikan waktu dua minggu lagi kepada seluruh konglomerat yang telah menandatangani perjanjian pengembalian bantuan likuiditas Bank Indonesia dengan jaminan aset ( MSAA ) , untuk secepatnya menyerahkan jaminan pribadi serta aset . Jika lewat dari tenggat

PREDICT: Pemerintah masih memberikan waktu dua minggu lagi kepada seluruh konglomerat yang telah menandatangani perjanjian pengembalian bantuan likuiditas Bank Indonesia dengan jaminan pribadi serta .

REFERENCE: Pemerintah memberikan tenggat 14 hari kepada para konglomerat penandatangan MSAA untuk menyerahkan aset . Jika mangkir , mereka bakal dihukum .

SCORE: {'rouge1': Score(precision=0.17391304347826086, recall=0.2222222222222222, fmeasure=0.1951219512195122), 'rougeL': Score(precision=0.17391304347826086, recall=0.2222222222222222, fmeasure=0.1951219512195122)}


In [22]:
# Percobaan dengan semua data Dev
for i in range(10):
    text = dev_data_convert[i]['clean_article']
    ref = dev_data_convert[i]['clean_summary']

    inputs = tokenizer(text, return_tensors="pt", truncation=True).to("cuda")
    summary_ids = model.generate(inputs["input_ids"], max_length=100)

    pred = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    print("PRED:", pred)
    print("REF :", ref)
    print("-"*50)

PRED: Pemerintah masih memberikan waktu dua minggu lagi kepada seluruh konglomerat , menyerahkan jaminan pribadi serta aset .
REF : Pemerintah memberikan tenggat 14 hari kepada para konglomerat penandatangan MSAA untuk menyerahkan aset . Jika mangkir , mereka bakal dihukum .
--------------------------------------------------
PRED: Pengasan 11 juta penduduk Indonesia positif terjangkit virus Hepatitis B . Bahkan , sepertiga di antaranya penderita kronis aktif yang bisa berakibat pengerasan hati dan kanker hati .
REF : Satu dari 20 orang Indonesia diperkirakan mengidap Hepatitis B . Tingginya angka tersebut lantaran rendahnya kesadaran dan pengetahuan tentang penyakit itu .
--------------------------------------------------
PRED: Tim Dokter Penilai untuk mendapat informasi yang tepat mengenai kondisi kesehatan Presiden Soeharto .
REF : Tim dokter yang bertugas memberikan informasi mengenai kondisi kesehatan Soeharto , perlu dibentuk . Keterangan tim ini bisa menentukan kehadiran Seoharto